## Ingestão BigQuery para Azure Blob Bronze

Notebook simplificado para execução local no Jupyter ou no Databricks. O fluxo carrega configurações, inicializa os clientes, exporta tabelas do BigQuery em Parquet e grava os arquivos no contêiner Bronze.


### 1. Dependências

Execute a célula abaixo apenas se o ambiente ainda não possuir as bibliotecas necessárias. Após instalar no Jupyter, reinicie o kernel.


In [17]:
%pip install azure-storage-blob google-cloud-bigquery google-cloud-bigquery-storage db-dtypes pyarrow python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [18]:
# ============================================================
# 1. IMPORTS
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional

import atexit
import json
import os
import sys
import tempfile

import google.cloud.bigquery as bigquery
import pandas as pd
import pyarrow.parquet as pq

from azure.core.credentials import AzureNamedKeyCredential
from azure.storage.blob import BlobServiceClient

try:
    from IPython.display import display
except ImportError:
    display = print


# ============================================================
# 2. CONFIGURAÇÕES GERAIS
# ============================================================

OVERWRITE = True

DEFAULT_GCP_PROJECT_ID = (
    "tough-medley-505300-k1"
)

DEFAULT_GCP_CREDENTIAL_FILE = (
    "tough-medley-505300-k1-164371097431.json"
)

# A data é calculada somente uma vez.
# O mesmo valor será usado em todos os arquivos.
DATA_EXECUCAO = datetime.now(
    timezone.utc
).strftime("%Y-%m-%d")


DEFAULT_TABLES = [
    (
        "basedosdados."
        "br_inep_avaliacao_alfabetizacao."
        "uf"
    ),
    (
        "basedosdados."
        "br_inep_avaliacao_alfabetizacao."
        "meta_alfabetizacao_brasil"
    ),
    (
        "basedosdados."
        "br_inep_avaliacao_alfabetizacao."
        "meta_alfabetizacao_uf"
    ),
    (
        "basedosdados."
        "br_inep_avaliacao_alfabetizacao."
        "meta_alfabetizacao_municipio"
    ),
    (
        "basedosdados."
        "br_inep_avaliacao_alfabetizacao."
        "municipio"
    ),
    (
        "basedosdados."
        "br_inep_avaliacao_alfabetizacao."
        "alunos"
    ),
]


# ============================================================
# 3. DETECTAR O AMBIENTE
# ============================================================

def detectar_databricks() -> bool:
    if os.getenv(
        "DATABRICKS_RUNTIME_VERSION"
    ):
        return True

    return globals().get(
        "dbutils"
    ) is not None


IS_DATABRICKS = detectar_databricks()

print(
    "✅ Ambiente:",
    (
        "Databricks"
        if IS_DATABRICKS
        else "local"
    ),
)

print(
    f"✅ Diretório atual: "
    f"{Path.cwd()}"
)

print(
    f"✅ Data da execução UTC: "
    f"{DATA_EXECUCAO}"
)


# ============================================================
# 4. LOCALIZAR A RAIZ DO PROJETO
# ============================================================

def localizar_raiz_projeto() -> Path:
    diretorio_atual = Path.cwd().resolve()

    for candidato in [
        diretorio_atual,
        *diretorio_atual.parents,
    ]:
        pasta_config = (
            candidato
            / "src"
            / "config"
        )

        arquivo_secrets = (
            pasta_config
            / "secrets.py"
        )

        if (
            pasta_config.is_dir()
            and arquivo_secrets.is_file()
        ):
            return candidato

    raise FileNotFoundError(
        "A raiz do projeto não foi encontrada. "
        "O arquivo src/config/secrets.py não existe "
        "no diretório atual nem nos diretórios superiores."
    )


BASE_DIR = localizar_raiz_projeto()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(BASE_DIR),
    )

print(
    f"✅ Raiz do projeto: "
    f"{BASE_DIR}"
)


# ============================================================
# 5. CARREGAR O .ENV LOCAL
# ============================================================

if not IS_DATABRICKS:
    from dotenv import load_dotenv

    ENV_PATH = (
        BASE_DIR
        / ".env"
    )

    if not ENV_PATH.is_file():
        raise FileNotFoundError(
            f"Arquivo .env não encontrado: "
            f"{ENV_PATH}"
        )

    ambiente_carregado = load_dotenv(
        dotenv_path=ENV_PATH,
        override=True,
    )

    if not ambiente_carregado:
        raise RuntimeError(
            f"Não foi possível carregar: "
            f"{ENV_PATH}"
        )

    print(
        f"✅ Ambiente local carregado: "
        f"{ENV_PATH}"
    )


# O import ocorre depois da configuração do sys.path
# e do carregamento do .env.
from src.config.secrets import get_secret


# ============================================================
# 6. CARREGAR SEGREDOS E CONFIGURAÇÕES
# ============================================================

def exigir_valor(
    valor: Optional[str],
    nome: str,
) -> str:
    valor_normalizado = (
        str(valor).strip()
        if valor is not None
        else ""
    )

    if not valor_normalizado:
        raise RuntimeError(
            f"{nome} não configurado."
        )

    return valor_normalizado


AZURE_STORAGE_ACCOUNT = exigir_valor(
    get_secret(
        secret_name="AZURE-STORAGE-ACCOUNT",
        env_name="AZURE_STORAGE_ACCOUNT",
        required=True,
    ),
    "AZURE_STORAGE_ACCOUNT",
)

AZURE_STORAGE_KEY = exigir_valor(
    get_secret(
        secret_name="AZURE-STORAGE-KEY",
        env_name="AZURE_STORAGE_KEY",
        required=True,
    ),
    "AZURE_STORAGE_KEY",
)

BRONZE_CONTAINER = exigir_valor(
    get_secret(
        secret_name="AZURE-CONTAINER-BRONZE",
        env_name="AZURE_CONTAINER_BRONZE",
        required=True,
    ),
    "AZURE_CONTAINER_BRONZE",
)

GOOGLE_CREDENTIALS_JSON = get_secret(
    secret_name=(
        "GOOGLE-APPLICATION-"
        "CREDENTIALS-JSON"
    ),
    env_name=(
        "GOOGLE_APPLICATION_"
        "CREDENTIALS_JSON"
    ),
    required=False,
)

GCP_PROJECT_ID = os.getenv(
    "GCP_PROJECT_ID",
    DEFAULT_GCP_PROJECT_ID,
).strip()

if not GCP_PROJECT_ID:
    raise RuntimeError(
        "GCP_PROJECT_ID não configurado."
    )


print(
    f"✅ Storage Account: "
    f"{AZURE_STORAGE_ACCOUNT}"
)

print(
    f"✅ Container Bronze: "
    f"{BRONZE_CONTAINER}"
)

print(
    f"✅ Projeto GCP: "
    f"{GCP_PROJECT_ID}"
)

print(
    f"✅ Storage Key presente: "
    f"{bool(AZURE_STORAGE_KEY)}"
)


# ============================================================
# 7. PASTA BRONZE LOCAL
# ============================================================

LOCAL_BRONZE_PATH = (
    BASE_DIR
    / "dados"
    / "bronze"
)

LOCAL_BRONZE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    f"✅ Pasta Bronze local: "
    f"{LOCAL_BRONZE_PATH.resolve()}"
)


# ============================================================
# 8. CONFIGURAR A CREDENCIAL GOOGLE
# ============================================================

TEMP_CREDENTIAL_PATH: Optional[str] = None


def remover_credencial_temporaria() -> None:
    global TEMP_CREDENTIAL_PATH

    if not TEMP_CREDENTIAL_PATH:
        return

    caminho = Path(
        TEMP_CREDENTIAL_PATH
    )

    if caminho.is_file():
        caminho.unlink()


atexit.register(
    remover_credencial_temporaria
)


def configurar_credencial_google() -> Path:
    global TEMP_CREDENTIAL_PATH

    if GOOGLE_CREDENTIALS_JSON:
        try:
            credencial_dict = json.loads(
                GOOGLE_CREDENTIALS_JSON
            )

        except json.JSONDecodeError as exc:
            raise RuntimeError(
                "A credencial Google armazenada no segredo "
                "não contém um JSON válido."
            ) from exc

        with tempfile.NamedTemporaryFile(
            mode="w",
            suffix=".json",
            delete=False,
            encoding="utf-8",
        ) as arquivo:
            json.dump(
                credencial_dict,
                arquivo,
                ensure_ascii=False,
            )

            TEMP_CREDENTIAL_PATH = (
                arquivo.name
            )

        caminho = Path(
            TEMP_CREDENTIAL_PATH
        )

        origem = (
            "Secret Scope ou variável de ambiente"
        )

    else:
        caminho = (
            BASE_DIR
            / "credenciais"
            / DEFAULT_GCP_CREDENTIAL_FILE
        )

        origem = "arquivo local"

    if not caminho.is_file():
        raise FileNotFoundError(
            f"Credencial Google não encontrada: "
            f"{caminho}"
        )

    os.environ[
        "GOOGLE_APPLICATION_CREDENTIALS"
    ] = str(caminho)

    print(
        f"✅ Credencial Google carregada de: "
        f"{origem}"
    )

    return caminho


# ============================================================
# 9. CARREGAR AS TABELAS
# ============================================================

def carregar_tabelas() -> List[str]:
    if not IS_DATABRICKS:
        return DEFAULT_TABLES.copy()

    databricks_utils = globals().get(
        "dbutils"
    )

    if databricks_utils is None:
        return DEFAULT_TABLES.copy()

    try:
        databricks_utils.widgets.text(
            "TABLES",
            "",
        )

        valor = (
            databricks_utils
            .widgets
            .get("TABLES")
            .strip()
        )

    except Exception as erro:
        print(
            "⚠️ Não foi possível acessar "
            f"o widget TABLES: {erro}"
        )

        return DEFAULT_TABLES.copy()

    if not valor:
        return DEFAULT_TABLES.copy()

    tabelas = [
        item.strip()
        for item in valor.split(",")
        if item.strip()
    ]

    return tabelas or DEFAULT_TABLES.copy()


GCP_CREDENTIAL_PATH = (
    configurar_credencial_google()
)

TABLES = carregar_tabelas()

print(
    f"✅ Tabelas configuradas: "
    f"{len(TABLES)}"
)

for tabela in TABLES:
    print(
        f"   • {tabela}"
    )


# ============================================================
# 10. INICIALIZAR OS CLIENTES
# ============================================================

def criar_cliente_bigquery():
    cliente = (
        bigquery.Client
        .from_service_account_json(
            str(GCP_CREDENTIAL_PATH),
            project=GCP_PROJECT_ID,
        )
    )

    print(
        "✅ Cliente BigQuery inicializado."
    )

    return cliente


def criar_cliente_blob():
    credencial = AzureNamedKeyCredential(
        name=AZURE_STORAGE_ACCOUNT,
        key=AZURE_STORAGE_KEY,
    )

    account_url = (
        f"https://{AZURE_STORAGE_ACCOUNT}"
        ".blob.core.windows.net"
    )

    cliente = BlobServiceClient(
        account_url=account_url,
        credential=credencial,
    )

    cliente.get_account_information()

    container_client = (
        cliente.get_container_client(
            container=BRONZE_CONTAINER
        )
    )

    container_client.get_container_properties()

    print(
        "✅ Cliente Azure Blob autenticado."
    )

    print(
        f"✅ Container acessível: "
        f"{BRONZE_CONTAINER}"
    )

    return cliente


bq_client = criar_cliente_bigquery()
blob_service_client = criar_cliente_blob()


# ============================================================
# 11. GERAR O NOME DATADO UMA ÚNICA VEZ
# ============================================================

def gerar_nome_datado(
    source_table: str,
) -> str:
    if not source_table:
        raise ValueError(
            "Tabela de origem não informada."
        )

    nome_tabela = source_table.rsplit(
        ".",
        maxsplit=1,
    )[-1]

    return (
        f"{DATA_EXECUCAO}_"
        f"{nome_tabela}.parquet"
    )


# ============================================================
# 12. EXPORTAR UMA TABELA
# ============================================================

def exportar_tabela(
    source_table: str,
) -> Dict[str, Any]:
    if not source_table:
        raise ValueError(
            "Tabela de origem não informada."
        )

    # O nome é gerado uma única vez.
    # O arquivo local e o blob usam o mesmo nome.
    nome_datado = gerar_nome_datado(
        source_table
    )

    query = (
        "SELECT * "
        f"FROM `{source_table}`"
    )

    print( f"▶ Consultando: {source_table}" )

    dataframe = (
        bq_client
        .query(
            query,
            location="US",
        )
        .to_dataframe()
    )

    if dataframe.empty:
        raise ValueError(
            f"Nenhum registro retornado por "
            f"{source_table}."
        )

    dataframe["_ingested_at"] = (
        datetime.now(
            timezone.utc
        )
    )

    dataframe["_source_table"] = (
        source_table
    )

    parquet_file = (
        LOCAL_BRONZE_PATH
        / nome_datado
    )

    dataframe.to_parquet(
        parquet_file,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )

    if not parquet_file.is_file():
        raise FileNotFoundError(
            f"Parquet não criado: "
            f"{parquet_file}"
        )

    tamanho_local = (
        parquet_file.stat().st_size
    )

    if tamanho_local <= 0:
        raise RuntimeError(
            f"O arquivo Parquet está vazio: "
            f"{parquet_file}"
        )

    print(
        f"✅ Parquet local: "
        f"{parquet_file.resolve()}"
    )

    blob_client = (
        blob_service_client
        .get_blob_client(
            container=BRONZE_CONTAINER,
            blob=nome_datado,
        )
    )

    with parquet_file.open(
        mode="rb"
    ) as arquivo:
        blob_client.upload_blob(
            data=arquivo,
            overwrite=OVERWRITE,
        )

    propriedades = (
        blob_client.get_blob_properties()
    )

    if propriedades.size != tamanho_local:
        raise RuntimeError(
            "Tamanho divergente entre "
            "o arquivo local e o Azure. "
            f"Local: {tamanho_local} bytes. "
            f"Azure: {propriedades.size} bytes."
        )

    resultado = {
        "success": True,
        "table": source_table,
        "container": BRONZE_CONTAINER,
        "blob": nome_datado,
        "local_file": str(
            parquet_file.resolve()
        ),
        "records": int(
            len(dataframe)
        ),
        "columns": int(
            len(dataframe.columns)
        ),
        "size_bytes": int(
            propriedades.size
        ),
    }

    print(
        f"✅ Blob enviado: "
        f"{nome_datado}"
    )

    print(
        f"✅ Registros: "
        f"{len(dataframe):,}"
    )

    print(
        f"✅ Colunas: "
        f"{len(dataframe.columns)}"
    )

    print(
        f"✅ Tamanho: "
        f"{propriedades.size / 1024:.2f} KB"
    )

    return resultado


# ============================================================
# 13. EXECUTAR A INGESTÃO DAS 6 TABELAS
# ============================================================

def executar_ingestao(
    tabelas: List[str],
) -> List[Dict[str, Any]]:
    resultados: List[Dict[str, Any]] = []
    falhas: List[Dict[str, str]] = []

    print()
    print(
        "========== INGESTÃO =========="
    )

    print(
        f"✅ Data UTC: "
        f"{DATA_EXECUCAO}"
    )

    print(
        f"✅ Total de tabelas: "
        f"{len(tabelas)}"
    )

    for indice, tabela in enumerate(
        tabelas,
        start=1,
    ):
        print()
        print(
            f"[{indice}/{len(tabelas)}] "
            f"{tabela}"
        )

        try:
            resultado = exportar_tabela(
                source_table=tabela,
            )

            resultados.append(
                resultado
            )

        except Exception as exc:
            erro = (
                f"{type(exc).__name__}: "
                f"{exc}"
            )

            falhas.append({
                "table": tabela,
                "error": erro,
            })

            print(
                f"❌ Falha: "
                f"{erro}"
            )

    print()
    print(
        "========== RESUMO =========="
    )

    print(
        f"✅ Sucessos: "
        f"{len(resultados)}"
    )

    print(
        f"❌ Falhas: "
        f"{len(falhas)}"
    )

    print(
        f"✅ Container: "
        f"{BRONZE_CONTAINER}"
    )

    if falhas:
        print()
        print(
            "Falhas encontradas:"
        )

        for falha in falhas:
            print(
                f"   • {falha['table']}: "
                f"{falha['error']}"
            )

        raise RuntimeError(
            f"Processo concluído com "
            f"{len(falhas)} falha(s)."
        )

    print(
        "✅ Todas as tabelas foram "
        "exportadas com sucesso."
    )

    return resultados


# ============================================================
# 14. EXIBIR 2 REGISTROS DE CADA TABELA BAIXADA
# ============================================================

def exibir_amostras(
    resultados: List[Dict[str, Any]],
    quantidade: int = 2,
) -> None:
    if quantidade <= 0:
        raise ValueError(
            "A quantidade deve ser maior que zero."
        )

    print()
    print(
        "========== AMOSTRAS =========="
    )

    if not resultados:
        print(
            "⚠️ Nenhuma tabela foi baixada."
        )

        return

    exibidas = 0
    falhas = []

    for indice, resultado in enumerate(
        resultados,
        start=1,
    ):
        caminho = Path(
            resultado["local_file"]
        )

        print()
        print(
            "=" * 100
        )

        print(
            f"[{indice}/{len(resultados)}] "
            f"{resultado['table']}"
        )

        print(
            "=" * 100
        )

        try:
            if not caminho.is_file():
                raise FileNotFoundError(
                    f"Arquivo não encontrado: "
                    f"{caminho}"
                )

            parquet = pq.ParquetFile(
                caminho
            )

            primeiro_lote = next(
                parquet.iter_batches(
                    batch_size=quantidade
                ),
                None,
            )

            print(
                f"✅ Arquivo local: "
                f"{caminho.name}"
            )

            print(
                f"✅ Blob no Azure: "
                f"{resultado['blob']}"
            )

            print(
                f"✅ Registros totais: "
                f"{parquet.metadata.num_rows:,}"
            )

            print(
                f"✅ Colunas: "
                f"{parquet.metadata.num_columns}"
            )

            if primeiro_lote is None:
                print(
                    "⚠️ Arquivo sem registros."
                )

                continue

            dataframe_preview = (
                primeiro_lote.to_pandas()
            )

            print(
                f"✅ Registros exibidos: "
                f"{len(dataframe_preview)}"
            )

            display(
                dataframe_preview
            )

            exibidas += 1

        except Exception as erro:
            mensagem = (
                f"{type(erro).__name__}: "
                f"{erro}"
            )

            falhas.append({
                "table": resultado["table"],
                "error": mensagem,
            })

            print(
                f"❌ Falha na visualização: "
                f"{mensagem}"
            )

    print()
    print(
        f"✅ Tabelas exibidas: "
        f"{exibidas}"
    )

    print(
        f"❌ Falhas na visualização: "
        f"{len(falhas)}"
    )

    if falhas:
        for falha in falhas:
            print(
                f"   • {falha['table']}: "
                f"{falha['error']}"
            )


# ============================================================
# 15. EXECUÇÃO
# ============================================================

resultados = executar_ingestao(
    TABLES
)

resumo = pd.DataFrame(
    resultados
)

print()
print(
    "========== RESUMO EM DATAFRAME =========="
)

display(
    resumo
)

exibir_amostras(
    resultados=resultados,
    quantidade=2,
)

✅ Ambiente: local
✅ Diretório atual: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/notebooks/bronze
✅ Data da execução UTC: 2026-10-19
✅ Raiz do projeto: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28
✅ Ambiente local carregado: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/.env
✅ Storage Account: stfiaptechchallenge2iast
✅ Container Bronze: bronze
✅ Projeto GCP: tough-medley-505300-k1
✅ Storage Key presente: True
✅ Pasta Bronze local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze
✅ Credencial Google carregada de: arquivo local
✅ Tabelas configuradas: 6
   • basedosdados.br_inep_avaliacao_alfabetizacao.uf
   • basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
   • basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf
   • basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio
   • basedosdados.br_inep_avaliacao_alfabetizacao.municipio
   • basedosdados.br_ine

,success,table,container,blob,local_file,records,columns,size_bytes
0,True,basedosdados.br_inep_avaliacao_alfabetizacao.uf,bronze,2026-10-19_uf.parquet,/home/linux/projetos/postech-2iast-fase3-model...,145,17,17895
1,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-19_meta_alfabetizacao_brasil.parquet,/home/linux/projetos/postech-2iast-fase3-model...,3,13,9462
2,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-19_meta_alfabetizacao_uf.parquet,/home/linux/projetos/postech-2iast-fase3-model...,81,14,12421
3,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-19_meta_alfabetizacao_municipio.parquet,/home/linux/projetos/postech-2iast-fase3-model...,10704,15,222848
4,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-19_municipio.parquet,/home/linux/projetos/postech-2iast-fase3-model...,23995,17,497733
5,True,basedosdados.br_inep_avaliacao_alfabetizacao.a...,bronze,2026-10-19_alunos.parquet,/home/linux/projetos/postech-2iast-fase3-model...,3867999,14,70795971



========== AMOSTRAS ==========

[1/6] basedosdados.br_inep_avaliacao_alfabetizacao.uf
✅ Arquivo local: 2026-10-19_uf.parquet
✅ Blob no Azure: 2026-10-19_uf.parquet
✅ Registros totais: 145
✅ Colunas: 17
✅ Registros exibidos: 2


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingested_at,_source_table
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-19 20:19:27.210905+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-19 20:19:27.210905+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf



[2/6] basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
✅ Arquivo local: 2026-10-19_meta_alfabetizacao_brasil.parquet
✅ Blob no Azure: 2026-10-19_meta_alfabetizacao_brasil.parquet
✅ Registros totais: 3
✅ Colunas: 13
✅ Registros exibidos: 2


...


[5/6] basedosdados.br_inep_avaliacao_alfabetizacao.municipio
✅ Arquivo local: 2026-10-19_municipio.parquet
✅ Blob no Azure: 2026-10-19_municipio.parquet
✅ Registros totais: 23,995
✅ Colunas: 17
✅ Registros exibidos: 2


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingested_at,_source_table
0,2023,1100031,2,3,69.1,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-19 20:19:43.475413+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
1,2023,1100072,2,3,58.2,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-19 20:19:43.475413+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...



[6/6] basedosdados.br_inep_avaliacao_alfabetizacao.alunos
✅ Arquivo local: 2026-10-19_alunos.parquet
✅ Blob no Azure: 2026-10-19_alunos.parquet
✅ Registros totais: 3,867,999
✅ Colunas: 14
✅ Registros exibidos: 2


,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,_ingested_at,_source_table
0,2023,1302603,60000712,13047236,1,2,2,0,0,0,NaN,NaN,2026-10-19 20:20:11.178001+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...
1,2023,2924058,60015855,29060897,1,2,3,0,0,0,NaN,NaN,2026-10-19 20:20:11.178001+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...



✅ Tabelas exibidas: 6
❌ Falhas na visualização: 0
